# 06 - Treino do Modelo com MLflow

No notebook 05 eu preparei a `gold_ml_features`, com as colunas que o modelo vai usar (medias moveis, volatilidade, retornos passados) e **dois targets** pra comparar: `target_retorno_prox_dia` e `target_volatilidade_prox_dia`. Agora vem a parte de treinar de verdade e acompanhar isso com o **MLflow**.

**Por que dois modelos?** O primeiro (retorno) eu ja sabia que ia sair fraco -- retorno diario de acao e proximo de ruido, isso e um resultado conhecido em financas, nao falha de implementacao. O segundo (volatilidade) testa um fenomeno diferente: volatilidade tem "memoria" (dia volatil tende a ser seguido de outro dia volatil), entao a expectativa e que esse modelo saia bem melhor. Treinar os dois e comparar mostra que eu entendo a diferenca entre o que da pra prever em financas e o que nao da -- isso vale mais que só um resultado bonito.

**Por que MLflow?** Cada modelo treinado vira um "run" registrado com parametros, metricas e o modelo salvo, sem precisar de setup extra no Databricks.

**Por que saio do Spark aqui?** A `gold_ml_features` tem menos de 9 mil linhas -- pequena pra qualquer computador rodar sem problema. scikit-learn e XGBoost trabalham com pandas/numpy, entao faz mais sentido converter pra pandas do que manter a complexidade do Spark pra um dataset desse tamanho.

O plano deste notebook:
1. Ler a `gold_ml_features` e converter pra pandas
2. Separar treino e teste respeitando a ordem do tempo
3. Treinar o modelo de **retorno** (baseline, esperado ser fraco)
4. Treinar o modelo de **volatilidade** (esperado ser melhor)
5. Comparar os dois lado a lado

In [0]:
%pip install mlflow xgboost scikit-learn

In [0]:
dbutils.library.restartPython()

## Passo 1 -- Lendo a feature store

O `restartPython()` limpa a memoria do notebook (por isso os pacotes novos entram em vigor), entao preciso ler os dados de novo aqui.

`.toPandas()` converte um DataFrame do Spark pra um DataFrame do pandas. So faco isso porque o dataset e pequeno -- se fossem milhoes de linhas, essa conversao ia estourar a memoria e eu precisaria treinar com Spark ML em vez disso.

In [0]:
df_pd = spark.table("b3_pipeline.gold_ml_features").toPandas()
df_pd = df_pd.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"Total de linhas: {len(df_pd)}")
df_pd.head()

## Passo 2 -- Separando treino e teste (respeitando o tempo)

Com serie temporal nao da pra usar `train_test_split` embaralhando as linhas aleatoriamente -- o modelo acabaria treinando com dados de datas futuras e sendo testado com datas passadas, o que se chama **vazamento de dados (data leakage)**. O resultado pareceria otimo no teste, mas seria inutil na vida real.

Por isso separo por data: pego uma data de corte, tudo antes vira treino, tudo depois vira teste -- exatamente como seria usar o modelo de verdade (so com o passado, prevendo o que ainda nao aconteceu).

In [0]:
# Pego todas as datas unicas em ordem, e escolho a que fica na posicao de 80% da lista
datas_unicas = sorted(df_pd["date"].unique())
data_corte = datas_unicas[int(len(datas_unicas) * 0.8)]

treino = df_pd[df_pd["date"] <= data_corte]
teste = df_pd[df_pd["date"] > data_corte]

print(f"Treino: {len(treino)} linhas | Teste: {len(teste)} linhas")
print(f"Data de corte: {data_corte}")

## Passo 3 -- Escolhendo features (X)

As features de entrada sao as mesmas pros dois modelos -- o que muda e so o `y` (o target). Deixei de fora `ticker`, `setor`, `date` e `close` por enquanto: sao texto ou data, e um modelo de regressao simples como esse precisa de numeros.

In [0]:
features = [
    "daily_return_pct",
    "media_movel_5d", "media_movel_10d", "volatilidade_5d",
    "retorno_lag1", "retorno_lag3", "retorno_lag5"
]

X_treino = treino[features]
X_teste = teste[features]

## Passo 4 -- Modelo 1: prevendo o retorno do dia seguinte

Uso o **XGBoost**, que e um modelo de *gradient boosting*: monta varias arvores de decisao pequenas em sequencia, cada uma corrigindo o erro que as anteriores deixaram passar.

`with mlflow.start_run():` abre um "run" -- tudo que acontece dentro desse bloco (parametros, metricas, modelo) fica registrado junto. Duas metricas de avaliacao:
- **MAE**: erro medio absoluto, em pontos percentuais de retorno
- **R2**: quanto o modelo explica da variacao do alvo (0 a 1, pode ser negativo se for pior que chutar a media)

Aviso antecipado: esse aqui deve sair fraco (R2 perto de zero). E o esperado -- guarda esse numero pra comparar com o proximo modelo.

In [0]:
import mlflow
import mlflow.xgboost
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

y_treino_retorno = treino["target_retorno_prox_dia"]
y_teste_retorno = teste["target_retorno_prox_dia"]

with mlflow.start_run(run_name="xgboost_retorno_prox_dia"):
    parametros = {
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.05,
        "random_state": 42
    }

    modelo_retorno = XGBRegressor(**parametros)
    modelo_retorno.fit(X_treino, y_treino_retorno)

    y_pred_retorno = modelo_retorno.predict(X_teste)
    mae_retorno = mean_absolute_error(y_teste_retorno, y_pred_retorno)
    r2_retorno = r2_score(y_teste_retorno, y_pred_retorno)

    for nome, valor in parametros.items():
        mlflow.log_param(nome, valor)
    mlflow.log_metric("mae", mae_retorno)
    mlflow.log_metric("r2", r2_retorno)
    mlflow.xgboost.log_model(modelo_retorno, "modelo")

    print(f"[Retorno] MAE: {mae_retorno:.4f}")
    print(f"[Retorno] R2: {r2_retorno:.4f}")

## Passo 5 -- Modelo 2: prevendo a volatilidade do dia seguinte

Mesma estrutura de treino, mesmas features -- so troco o `y` pra `target_volatilidade_prox_dia`. A diferenca nao esta no codigo, esta no fenomeno: volatilidade tem *volatility clustering* (dia volatil tende a ser seguido de outro dia volatil), entao existe padrao real pro modelo aprender, ao contrario do retorno.

Uso um `run_name` diferente pra esse ficar registrado como um experimento separado no MLflow, e da pra comparar os dois runs lado a lado na aba Experiments depois.

In [0]:
y_treino_vol = treino["target_volatilidade_prox_dia"]
y_teste_vol = teste["target_volatilidade_prox_dia"]

with mlflow.start_run(run_name="xgboost_volatilidade_prox_dia"):
    parametros = {
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.05,
        "random_state": 42
    }

    modelo_vol = XGBRegressor(**parametros)
    modelo_vol.fit(X_treino, y_treino_vol)

    y_pred_vol = modelo_vol.predict(X_teste)
    mae_vol = mean_absolute_error(y_teste_vol, y_pred_vol)
    r2_vol = r2_score(y_teste_vol, y_pred_vol)

    for nome, valor in parametros.items():
        mlflow.log_param(nome, valor)
    mlflow.log_metric("mae", mae_vol)
    mlflow.log_metric("r2", r2_vol)
    mlflow.xgboost.log_model(modelo_vol, "modelo")

    print(f"[Volatilidade] MAE: {mae_vol:.4f}")
    print(f"[Volatilidade] R2: {r2_vol:.4f}")

## Passo 6 -- Comparando os dois

Coloco os dois resultados lado a lado. A expectativa e que o R2 da volatilidade seja bem maior que o do retorno -- isso confirma, com os seus proprios dados, algo que a literatura de financas ja documenta: preco futuro e dificil de prever, risco futuro (volatilidade) e mais previsivel.

In [0]:
import pandas as pd

comparacao = pd.DataFrame({
    "modelo": ["retorno_prox_dia", "volatilidade_prox_dia"],
    "mae": [mae_retorno, mae_vol],
    "r2": [r2_retorno, r2_vol]
})

display(comparacao)

## Onde ver os resultados

No menu lateral do Databricks, clique em **Experiments** -- os dois runs (`xgboost_retorno_prox_dia` e `xgboost_volatilidade_prox_dia`) vao estar la, cada um com seus parametros, metricas e modelo salvo, prontos pra comparar.

Isso fecha a parte de ciencia de dados do projeto: fomos de dado bruto (Bronze) ate dois modelos treinados e rastreados (MLflow), testando duas hipoteses diferentes sobre o que da e o que nao da pra prever em series temporais financeiras.